# Import

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split

# split data

load dữ liệu

In [19]:
# ── 1. Load dữ liệu
df_train = pd.read_csv('dataset/mental_heath_unbanlanced.csv', encoding='utf-8')
df_test  = pd.read_csv('dataset/mental_health_combined_test.csv', encoding='utf-8')

print('Train columns:', df_train.columns.tolist())
print('Test  columns:', df_test.columns.tolist())

Train columns: ['Unique_ID', 'text', 'status']
Test  columns: ['text', 'status']


In [20]:

from tabulate import tabulate

# Đặt tên cột chứa văn bản và số ví dụ muốn hiển thị
text_col = "text"
max_examples = 5

print("\n--- KIỂM TRA DATA LEAKAGE ---")

# Chuẩn hóa text
train_texts = df_train[text_col].astype(str).str.strip().str.lower()
test_texts  = df_test[text_col].astype(str).str.strip().str.lower()

# Loại bỏ rỗng
train_texts = train_texts[train_texts != ""]
test_texts  = test_texts[test_texts != ""]

# Tập hợp unique
train_set = set(train_texts.unique())
test_set  = set(test_texts.unique())

# Tìm giao điểm
leakage_texts = train_set & test_set

# Thống kê
summary = [
    ["Train unique texts", len(train_set)],
    ["Test unique texts", len(test_set)],
    ["Leakage texts", len(leakage_texts)],
    ["Leak/Test Ratio (%)", f"{(len(leakage_texts)/len(test_set)*100):.2f}" if len(test_set) > 0 else "N/A"],
    ["Leak/Train Ratio (%)", f"{(len(leakage_texts)/len(train_set)*100):.2f}" if len(train_set) > 0 else "N/A"]
]
print(tabulate(summary, headers=["Thuộc tính", "Giá trị"], tablefmt="github"))

# In cảnh báo hoặc xác nhận
if leakage_texts:
    print(f"\nPhát hiện {len(leakage_texts):,} văn bản bị trùng lặp (leak) giữa tập Train và Test!")
    print(f"Tỷ lệ leak so với tập Test: {(len(leakage_texts)/len(test_set)*100):.2f}%")

    # Ví dụ minh họa
    print(f"\nVí dụ {min(max_examples, len(leakage_texts))} mẫu bị leak:")
    for i, txt in enumerate(list(leakage_texts)[:max_examples], 1):
        print(f"{i}. {txt[:120]}...")
else:
    print("\nKhông phát hiện data leak giữa Train và Test.")

print("----------------------------")


--- KIỂM TRA DATA LEAKAGE ---
| Thuộc tính           |   Giá trị |
|----------------------|-----------|
| Train unique texts   |  48929    |
| Test unique texts    |    992    |
| Leakage texts        |    496    |
| Leak/Test Ratio (%)  |     50    |
| Leak/Train Ratio (%) |      1.01 |

Phát hiện 496 văn bản bị trùng lặp (leak) giữa tập Train và Test!
Tỷ lệ leak so với tập Test: 50.00%

Ví dụ 5 mẫu bị leak:
1. alright... that's enoughi have tried to give life a chance but i can't it has only gotten worse in the last 6 years. i t...
2. i’m tryna get with a fireside girl but filler filler filler filler filler filler filler filler filler filler...
3. is there any reason to keep going?is it normal to be excluded from every group? i feel like no one really wanted to know...
4. things didn't get betteri posted here back in 2015 or 2016. at the time i was living in old trailer in the woods without...
5. is this legitin comments is the link. if i bought and took them would they actually kil

In [21]:
# ── Chuẩn hóa cột train, test
df_train = df_train[['text', 'status']]
df_test = df_test[['text', 'status']]

# ── Chuẩn hóa text + label
df_train['status'] = df_train['status'].str.lower().str.strip()
df_test['status'] = df_test['status'].str.lower().str.strip()

# ── Gộp dataset
df_all = pd.concat([df_train, df_test], ignore_index=True)

# ── Reset index
df_all = df_all.reset_index(drop=True)

print(f"\nTổng sau khi gộp: {len(df_all):,} mẫu")
print(df_all['status'].value_counts())


Tổng sau khi gộp: 50,604 mẫu
status
normal        18639
depression    14754
suicidal      11460
anxiety        5751
Name: count, dtype: int64


In [22]:
print("Số dòng trùng hoàn toàn:", df_all.duplicated().sum())

Số dòng trùng hoàn toàn: 1153


In [23]:
# Cột chứa văn bản và nhãn
text_col = "text"
label_col = "status"
max_examples = 5

print("\n===== KIỂM TRA TRÙNG LẶP & NHÃN =====")

# Tìm các text trùng lặp
duplicated_mask = df_all.duplicated(subset=[text_col], keep=False)
duplicated_df = df_all[duplicated_mask].sort_values(by=text_col)

if duplicated_df.empty:
    print(" Không tìm thấy nội dung trùng lặp.")
else:
    # Thống kê số lượng
    n_rows = len(duplicated_df)
    n_texts = duplicated_df[text_col].nunique()

    summary = [
        ["Số hàng trùng lặp", n_rows],
        ["Số 'text' duy nhất bị trùng", n_texts]
    ]
    print(tabulate(summary, headers=["Thuộc tính", "Giá trị"], tablefmt="github"))

    # Kiểm tra nhãn không nhất quán
    inconsistent_groups = []
    for text, group in duplicated_df.groupby(text_col):
        if group[label_col].nunique() > 1:
            inconsistent_groups.append(group)

    if not inconsistent_groups:
        print("\n Không tìm thấy sự thiếu nhất quán trong nhãn.")
    else:
        print(f"\n CẢNH BÁO: Có {len(inconsistent_groups)} nội dung '{text_col}' có nhãn '{label_col}' không nhất quán!")
        print(f"Ví dụ {min(max_examples, len(inconsistent_groups))} trường hợp:")

        for i, group_df in enumerate(inconsistent_groups[:max_examples], 1):
            print(f"\n{i}. Nội dung: {group_df[text_col].iloc[0]}")
            print(group_df[[text_col, label_col]])

print("----------------------------")



===== KIỂM TRA TRÙNG LẶP & NHÃN =====
| Thuộc tính                  |   Giá trị |
|-----------------------------|-----------|
| Số hàng trùng lặp           |      2179 |
| Số 'text' duy nhất bị trùng |      1016 |

 CẢNH BÁO: Có 10 nội dung 'text' có nhãn 'status' không nhất quán!
Ví dụ 5 trường hợp:

1. Nội dung: All this work, all this pressure that everyone puts on you to succeed. To go to a good college, get a good job, the normal things a lot of parents ask. All for what? I work my entire life and then what? Am I supposed to enjoy my ofttimes from studying or working being an ugly, socially awkward loser? Not able to talk to anyone, have friends; even when doing normally enjoyed things (video games, time off, etc.) all I can think about is how everyone else is probably enjoying their time with other people. Am I working for something or am I just working for the sake of working just because everyone tells me that is what I am supposed to do. Do people only tell you it gets better

In [24]:
# Đặt tên cột
TEXT_COL = "text"
LABEL_COL = "status"
MAX_EXAMPLES = 5

print("\n===== PHÂN TÍCH DUPLICATE & CONFLICT LABEL =====")

# Chuẩn hóa text
df_all["text_clean"] = (
    df_all[TEXT_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Tìm duplicate
dup_mask = df_all.duplicated(subset=["text_clean"], keep=False)
dup_df = df_all.loc[dup_mask, ["text_clean", LABEL_COL]].copy()

total_dup_rows = len(dup_df)
total_dup_texts = dup_df["text_clean"].nunique()

if dup_df.empty:
    print(" Không phát hiện duplicate")
else:
    # Thống kê
    summary = [
        ["Số dòng duplicate", total_dup_rows],
        ["Số text duplicate", total_dup_texts],
    ]
    print(tabulate(summary, headers=["Thuộc tính", "Giá trị"], tablefmt="github"))

    # Kiểm tra conflict label
    conflict_df = (
        dup_df.groupby("text_clean", sort=False)[LABEL_COL]
        .nunique()
        .reset_index(name="n_labels")
    )
    conflict_texts = conflict_df[conflict_df["n_labels"] > 1]
    n_conflicts = len(conflict_texts)

    print(f"\nSố text conflict label : {n_conflicts:,}")
    conflict_rate = (n_conflicts / total_dup_texts * 100) if total_dup_texts > 0 else 0
    print(f"Tỷ lệ conflict         : {conflict_rate:.2f}%")

    # Hiển thị ví dụ conflict
    if n_conflicts > 0:
        print(f"\n Ví dụ {min(MAX_EXAMPLES, n_conflicts)} trường hợp conflict:")
        sample_conflicts = (
            dup_df[dup_df["text_clean"].isin(conflict_texts["text_clean"])]
            .drop_duplicates()
            .head(MAX_EXAMPLES)
        )
        print(sample_conflicts)
    else:
        print("\n Không có conflict label")

print("\nHoàn tất kiểm tra duplicate & conflict")
print("--------------------------------------------------")



===== PHÂN TÍCH DUPLICATE & CONFLICT LABEL =====
| Thuộc tính        |   Giá trị |
|-------------------|-----------|
| Số dòng duplicate |      2204 |
| Số text duplicate |      1025 |

Số text conflict label : 13
Tỷ lệ conflict         : 1.27%

 Ví dụ 5 trường hợp conflict:
                                              text_clean      status
7637   all this work, all this pressure that everyone...    suicidal
7638   all this work, all this pressure that everyone...  depression
10984  i have been at home the past year... quite lit...    suicidal
10989  i have been at home the past year... quite lit...  depression
19080  my twin betrayed me and it cost me some of my ...    suicidal

Hoàn tất kiểm tra duplicate & conflict
--------------------------------------------------


In [25]:

initial_rows = len(df_all)

print("\n===== LÀM SẠCH DỮ LIỆU =====")

# 1. Loại bỏ các văn bản có nhãn 'status' không nhất quán
inconsistent_texts = []
for text, group in df_all.groupby("text"):
    if group["status"].nunique() > 1:
        inconsistent_texts.append(text)

if inconsistent_texts:
    df_all_cleaned = df_all[~df_all["text"].isin(inconsistent_texts)].copy()
    print(f" Đã loại bỏ {len(inconsistent_texts)} văn bản có nhãn không nhất quán.")
else:
    df_all_cleaned = df_all.copy()
    print(" Không tìm thấy văn bản có nhãn không nhất quán.")

# 2. Loại bỏ các hàng trùng lặp hoàn toàn (cả 'text' và 'status')
before_drop = len(df_all_cleaned)
df_all_cleaned.drop_duplicates(subset=["text", "status"], inplace=True)
after_drop = len(df_all_cleaned)
print(f" Đã loại bỏ {before_drop - after_drop} hàng trùng lặp hoàn toàn.")

removed_rows = initial_rows - len(df_all_cleaned)

# Thống kê sau làm sạch
summary = [
    ["Số hàng ban đầu", initial_rows],
    ["Số hàng sau làm sạch", len(df_all_cleaned)],
    ["Tổng số hàng đã loại bỏ", removed_rows],
]
print(tabulate(summary, headers=["Thuộc tính", "Giá trị"], tablefmt="github"))

# Cập nhật df_all
df_all = df_all_cleaned

# 3. Kiểm tra lại duplicate sau làm sạch
print("\n--- Kiểm tra lại trùng lặp nội dung 'text' ---")
dup_mask_cleaned = df_all.duplicated(subset=["text"], keep=False)
dup_df_cleaned = df_all[dup_mask_cleaned].sort_values(by="text")

if dup_df_cleaned.empty:
    print(" Không tìm thấy nội dung 'text' trùng lặp sau khi làm sạch.")
else:
    print(f" Tìm thấy {len(dup_df_cleaned)} hàng trùng lặp. "
          f"Tổng số 'text' duy nhất bị trùng: {dup_df_cleaned['text'].nunique()}")

    # Kiểm tra conflict label sau làm sạch
    inconsistent_labels_cleaned = []
    for text, group in dup_df_cleaned.groupby("text"):
        if group["status"].nunique() > 1:
            inconsistent_labels_cleaned.append(group)

    if not inconsistent_labels_cleaned:
        print(" Không có conflict label trong các text trùng lặp sau làm sạch.")
    else:
        print(f" Có {len(inconsistent_labels_cleaned)} nội dung 'text' vẫn conflict label:")
        for group_df in inconsistent_labels_cleaned[:5]:  # hiển thị tối đa 5 ví dụ
            print(f"\nNội dung: '{group_df['text'].iloc[0]}'")
            print(group_df[["text", "status"]])

# Phân phối nhãn sau làm sạch
print("\n--- Phân phối nhãn 'status' sau làm sạch ---")
print(df_all["status"].value_counts())

print("--------------------------------------------------")



===== LÀM SẠCH DỮ LIỆU =====
 Đã loại bỏ 10 văn bản có nhãn không nhất quán.
 Đã loại bỏ 1149 hàng trùng lặp hoàn toàn.
| Thuộc tính              |   Giá trị |
|-------------------------|-----------|
| Số hàng ban đầu         |     50604 |
| Số hàng sau làm sạch    |     49431 |
| Tổng số hàng đã loại bỏ |      1173 |

--- Kiểm tra lại trùng lặp nội dung 'text' ---
 Không tìm thấy nội dung 'text' trùng lặp sau khi làm sạch.

--- Phân phối nhãn 'status' sau làm sạch ---
status
normal        18151
depression    14507
suicidal      11200
anxiety        5573
Name: count, dtype: int64
--------------------------------------------------


In [26]:
from tabulate import tabulate

# ── 3. Tách X, y
X = df_all['text']
y = df_all['status']

# ── 4. Chia train / val / test (70 / 10 / 20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,   # 0.125 của 0.8 = 0.1 → val = 10%
    random_state=42,
    stratify=y_temp
)

# ── 5. Kiểm tra kết quả
summary_split = [
    ["Train", len(X_train)],
    ["Val", len(X_val)],
    ["Test", len(X_test)],
    ["Tổng cộng", len(df_all)]
]
print("\n===== KẾT QUẢ CHIA TẬP =====")
print(tabulate(summary_split, headers=["Tập dữ liệu", "Số mẫu"], tablefmt="github"))

# Phân phối nhãn
def show_distribution(y, name):
    dist = y.value_counts(normalize=True).round(3)
    print(f"\n--- Phân phối nhãn {name} ---")
    print(dist)

show_distribution(y_train, "Train")
show_distribution(y_val, "Val")
show_distribution(y_test, "Test")



===== KẾT QUẢ CHIA TẬP =====
| Tập dữ liệu   |   Số mẫu |
|---------------|----------|
| Train         |    34601 |
| Val           |     4943 |
| Test          |     9887 |
| Tổng cộng     |    49431 |

--- Phân phối nhãn Train ---
status
normal        0.367
depression    0.293
suicidal      0.227
anxiety       0.113
Name: proportion, dtype: float64

--- Phân phối nhãn Val ---
status
normal        0.367
depression    0.294
suicidal      0.227
anxiety       0.113
Name: proportion, dtype: float64

--- Phân phối nhãn Test ---
status
normal        0.367
depression    0.294
suicidal      0.227
anxiety       0.113
Name: proportion, dtype: float64


In [27]:

# ── Train
pd.DataFrame({
    'text': X_train,
    'status': y_train
}).to_csv('train.csv', index=False, encoding='utf-8')

# ── Validation
pd.DataFrame({
    'text': X_val,
    'status': y_val
}).to_csv('val.csv', index=False, encoding='utf-8')

# ── Test
pd.DataFrame({
    'text': X_test,
    'status': y_test
}).to_csv('test.csv', index=False, encoding='utf-8')

print('\n Saved files:')
print('- train.csv')
print('- val.csv')
print('- test.csv')


 Saved files:
- train.csv
- val.csv
- test.csv
